# Train DRPO: Adaptive and Matched Uniform

Training only: official GSM8K train, DeepSeek-R1-Distill-Qwen-1.5B, 512 responses per arm,
4096 new tokens, m=4/M=8, mu=0/sigma=5, multiplier=2. Both arms start independently
from the same pinned original model. The other training notebook shares the pinned
configuration and data, but trains a different RL objective.

Adaptive runs first. The "fixed" arm preserves the previous matched near-uniform
baseline: same K training questions/order as this algorithm's adaptive arm;
floor(512/K) responses each plus one extra to randomly selected questions. It is
not a prespecified fixed-6 baseline. At intermediate budgets the question coverage
and actual optimizer step count may differ; all counts are recorded.

Snapshots are saved AFTER updates at the first completed question reaching each
threshold 128/256/384/512. Folder names identify targets; READY.json records actual
rollouts (e.g. 131). No forced stopping is added at intermediate thresholds. Only
the final 512-response budget can cut a question below m. Use actual counts on plots.

DRPO preserves the original decoupled positive-length/negative objective and sampled old-policy KL penalty. The logged loss KL is pre-update; the separate post-update sampled old/new diagnostic measures local update drift, not cumulative drift from the base.

All-correct/all-wrong groups can skip optimizer steps. Neither notebook evaluates
test questions. Training does not support mid-arm resume; incomplete runs need a
new EXPERIMENT_NAME. Completed checkpoints remain available for evaluation.
Full checkpoints consume several GB each; 16 snapshots across both algorithms can
require tens of GB of Drive storage. Optimizer state is not saved.


## 1. Install dependencies


In [ ]:
import subprocess
import sys
# Keep the CUDA-enabled PyTorch supplied by Colab.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.1", "datasets==4.3.0", "accelerate==1.11.0",
    "peft==0.17.1", "math-verify==0.8.0", "pandas>=2,<3", "matplotlib>=3.8"])


## 2. Mount Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import os
import sys
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/GRPO_DRPO_Checkpoints_Project")
EXPERIMENT_NAME = "seed42_gsm8k_512_grpo_drpo_v11"
assert (PROJECT / "rl_project/training.py").is_file(), "Upload the complete project folder."
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
RUN_ROOT = PROJECT / "runs" / EXPERIMENT_NAME
print("Experiment:", RUN_ROOT)


## 3. Configure and freeze the experiment


In [ ]:
ALGORITHM = "drpo"
SEED = 42
TOTAL_ROLLOUTS = 512
CHECKPOINT_ROLLOUTS = [128, 256, 384, 512]
MIN_ANSWERS = 4
MAX_ANSWERS = 8
SAMPLING_MU = 0.0
SAMPLING_SIGMA = 5.0
SAMPLING_MULTIPLIER = 2.0
MAX_NEW_TOKENS = 4096

cfg = json.loads((PROJECT / "config.json").read_text())
cfg.update(seed=SEED, total_rollout_budget=TOTAL_ROLLOUTS,
    checkpoint_rollouts=CHECKPOINT_ROLLOUTS, min_rollouts=MIN_ANSWERS,
    max_rollouts=MAX_ANSWERS, sampling_mu=SAMPLING_MU, sampling_sigma=SAMPLING_SIGMA,
    sampling_multiplier=SAMPLING_MULTIPLIER, max_new_tokens=MAX_NEW_TOKENS)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime. A100 is recommended for full-parameter training.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("Select a BF16-capable GPU, such as L4 or A100, for this full-parameter configuration.")
print("GPU:", torch.cuda.get_device_name(0))
print("Each arm generates", TOTAL_ROLLOUTS, "responses, starting independently from the base.")
print("Settings must be identical in both training notebooks for the same experiment name.")
from rl_project.training import configure
RUN_ROOT = configure(PROJECT, EXPERIMENT_NAME, cfg)


## 4. Train adaptive first


In [ ]:
def train_arm(mode):
    command = [sys.executable, "-u", "-m", "rl_project.training", "--root", str(RUN_ROOT),
               "--algorithm", ALGORITHM, "--mode", mode]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

train_arm("adaptive")


## 5. Train the matched near-uniform baseline


In [ ]:
train_arm("fixed")


## 6. Inspect saved checkpoints


In [ ]:
for mode in ("adaptive", "fixed"):
    arm = RUN_ROOT / ALGORITHM / mode
    print(mode, json.loads((arm / "summary.json").read_text()))
    print(json.dumps(json.loads((arm / "checkpoint_index.json").read_text()), indent=2))
print("Open independent notebooks under evaluation_notebooks/ to evaluate any ready checkpoint.")
